# Practical Exercise: Experimenting with tokenization and vectorization

In this first exercise, the goal will be to build your own text vectorizer including:
* Preprocessing
* Tokenization
* Vectorization

Let us start with tokenization and, in particular, the most common case which is word tokenization.

## Tokenizer

Consider the following sentence:

In [1]:
sentence = "the cat eats the mouse"

Here tokens will be words. Write a piece of code that tokenizes the sentence `sentence`.

In [2]:
res = sentence.split(" ")
print(res)

['the', 'cat', 'eats', 'the', 'mouse']


Now consider that tokens are characters and tokenize `sentence` accordingly. Note: consider the whitespace as a character in itself.

In [3]:
a = []
for word in sentence:
    for i in word:
        a.append(i)
print(a)

['t', 'h', 'e', ' ', 'c', 'a', 't', ' ', 'e', 'a', 't', 's', ' ', 't', 'h', 'e', ' ', 'm', 'o', 'u', 's', 'e']


Character and word tokenization are two basic but very popular tokenization strategies. Now that we know how to apply these two fundamentals, let us move on to vectorization.

## Vectorization

Remember, vectorization consists in building a vector space in which we can represent a corpus of texts. This means that it aims at converting any sequence of tokens into a vector, such that vectors are comparable in a meaningful manner. Let us practice to familiarize with this concept.

We will consider word tokenization and will start by implementing a Count Vectorizer (BOW). Count Vectorizer is a vectorizer for which each variable/component is associated to a token, and its value is the number of occurence of the token in the text.

Let us consider a corpus of two texts:

In [4]:
corpus = ['a dog eats a bone','a cat is playing with a dog']

Start by vectorizing the first sentence alone. Your goal will be to (1) obtain a vector that represents your text (you can call it `text_vector`) with its components being word counts and (2) a list that contains the variable names (let us call it `vocabulary`).
In this example, we will sort the variables alphabetically.

In [5]:
# step 1
tokens = corpus[0].split(" ")
print(tokens)

# step 2
counts = {}
for word in tokens:
    if word not in counts:
        counts[word] = 1
    else:
        counts[word] += 1
print(counts)

# step 3
vocabulary = sorted(counts)
text_vector = [counts[word] for word in vocabulary]

# step 4
print(text_vector)
print(vocabulary)

['a', 'dog', 'eats', 'a', 'bone']
{'a': 2, 'dog': 1, 'eats': 1, 'bone': 1}
[2, 1, 1, 1]
['a', 'bone', 'dog', 'eats']


Make sure the variables contain the following values before moving on to the next question:
* `text_vector`:
```
array([2, 1, 1, 1])
```
* `vocabulary`:
```
['a', 'bone', 'dog', 'eats']
```

Now, turn your implementation into a function called `vectorize_single_text` that takes as input a text and returns the associated `text_vector` and `vocabulary`.

In [6]:
def vectorize_single_text(text):
    tokens = text.split(" ")
    #print(tokens)

    counts = {}
    for word in tokens:
        if word not in counts:
            counts[word] = 1
        else:
            counts[word] += 1
    #print(counts)
    
    vocabulary = sorted(counts)
    text_vector = [counts[word] for word in vocabulary]
    return text_vector,vocabulary

Use your function to featurize the first text again (`text_vector1`,`vocabulary1`) and then the second text (`text_vector2`,`vocabulary2`)

In [7]:
text_vector1, vocabulary1 = vectorize_single_text(corpus[0])

In [8]:
text_vector2, vocabulary2 = vectorize_single_text(corpus[1])

In [9]:
print(text_vector1, vocabulary1)
print(text_vector2, vocabulary2)

[2, 1, 1, 1] ['a', 'bone', 'dog', 'eats']
[2, 1, 1, 1, 1, 1] ['a', 'cat', 'dog', 'is', 'playing', 'with']


Give two reasons why you cannot compare `text_vector1` and `text_vector2`:


How can you solve this issue ?


Let us consider our final corpus with the three sentences (`sentence` and the two sentences from `corpus`) and build a proper Count Vectorizer.

In [10]:
corpus += [sentence]

In [11]:
corpus

['a dog eats a bone', 'a cat is playing with a dog', 'the cat eats the mouse']

In [12]:
from collections import Counter

# step 1: tokenize each sentence
res = [sentence.split(" ") for sentence in corpus]
print(res)

# step 2: build associated counter
cnt = [Counter(tokens) for tokens in res]
print(cnt)

# step 3: join all vocabularies
global_vocabulary = {w for tokens in res for w in tokens}
print(global_vocabulary)

# step 4: build all vectors
vectors = [[c[word] for word in global_vocabulary] for c in cnt]
print(vectors)


[['a', 'dog', 'eats', 'a', 'bone'], ['a', 'cat', 'is', 'playing', 'with', 'a', 'dog'], ['the', 'cat', 'eats', 'the', 'mouse']]
[Counter({'a': 2, 'dog': 1, 'eats': 1, 'bone': 1}), Counter({'a': 2, 'cat': 1, 'is': 1, 'playing': 1, 'with': 1, 'dog': 1}), Counter({'the': 2, 'cat': 1, 'eats': 1, 'mouse': 1})]
{'cat', 'dog', 'is', 'bone', 'playing', 'the', 'a', 'with', 'eats', 'mouse'}
[[0, 1, 0, 1, 0, 0, 2, 0, 1, 0], [1, 1, 1, 0, 1, 0, 2, 1, 0, 0], [1, 0, 0, 0, 0, 2, 0, 0, 1, 1]]


In [13]:
print(vectors)
print(global_vocabulary)

[[0, 1, 0, 1, 0, 0, 2, 0, 1, 0], [1, 1, 1, 0, 1, 0, 2, 1, 0, 0], [1, 0, 0, 0, 0, 2, 0, 0, 1, 1]]
{'cat', 'dog', 'is', 'bone', 'playing', 'the', 'a', 'with', 'eats', 'mouse'}


Now all the vectors have the same size and are comparable, let us apply a similarity measure (cosine similarity) to compare them pair by pair and answer some questions.

Note: the cosine similarity function which is defined as the dot product divided by the product of the L2-norms.

* Which sentences are the most similar?
* Can you explain why?
* Do you think the similarity order should be different? If yes, think about how you could make this happen?

In [14]:
def inner_prod(X, Y):
   sum_prod = 0
   for x_i, y_i in zip(X, Y):
       sum_prod += x_i * y_i
   return sum_prod


def magnitude(X):
   sum_squares = 0
   for x_i in X:
       sum_squares += x_i ** 2
   return sum_squares**0.5


def cosine_similarity(vec1, vec2):
    return inner_prod(vec1, vec2) / (magnitude(vec1) * magnitude(vec2))


In [15]:
for i in range(len(vectors)):
    for j in range(len(vectors)):
        print(i,j,cosine_similarity(vectors[i],vectors[j]))

0 0 0.9999999999999999
0 1 0.629940788348712
0 2 0.14285714285714285
1 0 0.629940788348712
1 1 1.0
1 2 0.1259881576697424
2 0 0.14285714285714285
2 1 0.1259881576697424
2 2 0.9999999999999999


In [16]:
import numpy as np
from sklearn.metrics import pairwise_distances
for i in range(len(vectors)):
    for j in range(len(vectors)):
        print(i,j,pairwise_distances(np.array([vectors[i]]),np.array([vectors[j]]), metric='manhattan'))

0 0 [[0.]]
0 1 [[6.]]
0 2 [[8.]]
1 0 [[6.]]
1 1 [[0.]]
1 2 [[10.]]
2 0 [[8.]]
2 1 [[10.]]
2 2 [[0.]]


The two most similar sentences are 'a dog eats a bone' and 'a cat is playing with a dog' because they have more words in common than with the third sentence 'the cat eats the mouse'.

One could argue that 'a dog eats a bone' and 'the cat eats the mouse' should be more similar. One way to achieve this results would be to avoid counting similarity on 'stop-words' like 'a', 'the', etc.

## Managing stop words

Below is a list of a with 5 English stop words. Let us try to add stop words removal to our tokenizer/vectorizer and compute similarity again.

In [17]:
stop_words = ['a', 'or', 'with', 'the', 'is']

In [18]:
#step 1: tokenize each sentence and remove stop words
res = [sentence.split(" ") for sentence in corpus]
res = [[word for word in tokens if word not in stop_words] for tokens in res]
print(res)

#step 2: build associated counter
cnt = [Counter(tokens) for tokens in res]
print(cnt)

#step 3: join all vocabularies
global_vocabulary = {word for tokens in res for word in tokens}
print(global_vocabulary)

#step 4: build all vectors
vectors = [[c[word] for word in global_vocabulary] for c in cnt]
print(vectors)

[['dog', 'eats', 'bone'], ['cat', 'playing', 'dog'], ['cat', 'eats', 'mouse']]
[Counter({'dog': 1, 'eats': 1, 'bone': 1}), Counter({'cat': 1, 'playing': 1, 'dog': 1}), Counter({'cat': 1, 'eats': 1, 'mouse': 1})]
{'cat', 'dog', 'bone', 'playing', 'eats', 'mouse'}
[[0, 1, 1, 0, 1, 0], [1, 1, 0, 1, 0, 0], [1, 0, 0, 0, 1, 1]]


In [19]:
print(vectors)
print(global_vocabulary)

[[0, 1, 1, 0, 1, 0], [1, 1, 0, 1, 0, 0], [1, 0, 0, 0, 1, 1]]
{'cat', 'dog', 'bone', 'playing', 'eats', 'mouse'}


In [20]:
for i in range(len(vectors)):
    for j in range(len(vectors)):
        print(i,j,cosine_similarity(vectors[i],vectors[j]))

0 0 1.0000000000000002
0 1 0.33333333333333337
0 2 0.33333333333333337
1 0 0.33333333333333337
1 1 1.0000000000000002
1 2 0.33333333333333337
2 0 0.33333333333333337
2 1 0.33333333333333337
2 2 1.0000000000000002


In [21]:
# compute similarities with pairwise_distances
import numpy as np
from sklearn.metrics import pairwise_distances
for i in range(len(vectors)):
    for j in range(len(vectors)):
        print(i,j,pairwise_distances(np.array([vectors[i]]),np.array([vectors[j]]), metric='manhattan'))

0 0 [[0.]]
0 1 [[4.]]
0 2 [[4.]]
1 0 [[4.]]
1 1 [[0.]]
1 2 [[4.]]
2 0 [[4.]]
2 1 [[4.]]
2 2 [[0.]]


Note how the similarity scores have changed when removing stop-words.

## Adding TF-IDF

As explained earlier, an alternative to Count vectorization is TF-IDF vectorization. Instead of the count, TF-IDF relies on a more relevant statistics which takes into account the frequency of the token in the text and its frequency on the overall corpus. This allows to give importance to the elements that contrast the text with the rest of the corpus.

Modify your previous implementation to obtain TF-IDF vectors instead of Count vectors.

In [22]:
#reuse same tokens and vocabulary from previously
#build all vectors with tfidf instead of count
import math


res = [sentence.split(" ") for sentence in corpus]
res2 = [sentence.split(" ") for sentence in corpus]
res = [[word for word in tokens if word not in stop_words] for tokens in res]
res=[j for sub in res for j in sub]
print(global_vocabulary)
print(res)
print(res2)
def tf(t,d):
    a=Counter(d)
    return a[t]/len(d)

def idf(t,D):
    c=0
    for i in D:
        c+=Counter(i)[t]
        
    return math.log(len(D)/c)
    
    
print(tf('dog',res))
print(idf('dog',res2))

{'cat', 'dog', 'bone', 'playing', 'eats', 'mouse'}
['dog', 'eats', 'bone', 'cat', 'playing', 'dog', 'cat', 'eats', 'mouse']
[['a', 'dog', 'eats', 'a', 'bone'], ['a', 'cat', 'is', 'playing', 'with', 'a', 'dog'], ['the', 'cat', 'eats', 'the', 'mouse']]
0.2222222222222222
0.4054651081081644


## Using sklearn implementation

Count vectorization and TF-IDF vectorization are both implemented in standard ML library sklearn under `sklearn.feature_extraction.text.CountVectorizer` and `sklearn.feature_extraction.text.TfidfVectorizer`. Visit the documentation of these two classes to try to obtain the same similarities you obtain with your own implementations.

In [23]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [24]:
vectorizer = CountVectorizer()
vectors = vectorizer.fit_transform(corpus)

In [25]:
vectorizer.vocabulary_

{'dog': 2,
 'eats': 3,
 'bone': 0,
 'cat': 1,
 'is': 4,
 'playing': 6,
 'with': 8,
 'the': 7,
 'mouse': 5}

In [26]:
vectors.toarray()

array([[1, 0, 1, 1, 0, 0, 0, 0, 0],
       [0, 1, 1, 0, 1, 0, 1, 0, 1],
       [0, 1, 0, 1, 0, 1, 0, 2, 0]])

In [27]:
# compute and compare similarities

In [28]:
vectorizer = TfidfVectorizer() # TFIDF
vectors = vectorizer.fit_transform(corpus)

In [29]:
vectorizer.vocabulary_

{'dog': 2,
 'eats': 3,
 'bone': 0,
 'cat': 1,
 'is': 4,
 'playing': 6,
 'with': 8,
 'the': 7,
 'mouse': 5}

In [30]:
vectors.toarray()

array([[0.68091856, 0.        , 0.51785612, 0.51785612, 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.37302199, 0.37302199, 0.        , 0.49047908,
        0.        , 0.49047908, 0.        , 0.49047908],
       [0.        , 0.30650422, 0.        , 0.30650422, 0.        ,
        0.40301621, 0.        , 0.80603242, 0.        ]])

## N-grams

Let us now consider another corpus:

In [31]:
corpus2 = ['the cat eats the dog', 'the dog eats the cat', 'I love fries but I hate brocoli', 'I love brocoli but I hate fries']

Try vectorizing this corpus and analyze the vectors:

In [32]:
vectorizer2 = CountVectorizer()
vectorizer2.fit_transform(corpus2)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 18 stored elements and shape (4, 9)>

In [33]:
vectors.toarray()

array([[0.68091856, 0.        , 0.51785612, 0.51785612, 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.37302199, 0.37302199, 0.        , 0.49047908,
        0.        , 0.49047908, 0.        , 0.49047908],
       [0.        , 0.30650422, 0.        , 0.30650422, 0.        ,
        0.40301621, 0.        , 0.80603242, 0.        ]])

* Is there something wrong with the obtained vectors?
* How could you solve this?

Let us change the n-gram parameter so that our tokens are now both words and bi-gram (two successive words) and analyze again the vectors.

In [34]:
vectorizer2 = CountVectorizer(analyzer='word', ngram_range=(1, 2))
vectors = vectorizer2.fit_transform(corpus2)

In [35]:
vectors.toarray()

array([[0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1],
       [0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1],
       [1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0],
       [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0]])

Does this solve the problem?

## Going further with lemmatization, stemming

Visit https://www.nltk.org/howto/stem.html to see how to use `PorterStemmer` and `SnowballStemmer` from NLTK library.

Use it to stem sentences from `corpus3`.

In [36]:
corpus3 = ['dogs eat cats', 'a cat eats a dog', 'I love brocolis', 'I loved brocoli']

In [37]:
!pip install nltk

In [38]:
from nltk.stem import *


In [39]:
stemmer = PorterStemmer()
res = [sentence.split(" ") for sentence in corpus3]
res=[j for sub in res for j in sub]
print(res)
singles = [stemmer.stem(i) for i in res]
print(singles)
print(' '.join(singles))

['dogs', 'eat', 'cats', 'a', 'cat', 'eats', 'a', 'dog', 'I', 'love', 'brocolis', 'I', 'loved', 'brocoli']
['dog', 'eat', 'cat', 'a', 'cat', 'eat', 'a', 'dog', 'i', 'love', 'brocoli', 'i', 'love', 'brocoli']
dog eat cat a cat eat a dog i love brocoli i love brocoli


In [40]:
stemmer = SnowballStemmer("french")
print(stemmer.stem("running"))

running


## Still have some time?

* Try using nltk's tokenizers
* Try using spacy English tokenizer (stem, stop-words removal)